In [8]:
import numpy as np
import sympy as sp
import math
import plotly.graph_objects as go

def newton(f, x0, e1, e2, gamma, M):
    x = x0
    x1, x2 = sp.symbols('x1, x2')
    fun = sp.lambdify([x1, x2], f, "math")

    trajectory = [x0]
    values = [fun(x[0], x[1])]

    k = 0
    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    hessian = sp.hessian(f, [x1, x2])
    x1d_lamb = sp.lambdify([x1, x2], x1d, "math")
    x2d_lamb = sp.lambdify([x1, x2], x2d, "math")
    
    x = x.astype(float)
    while k < M:
        c_grad = np.array([x1d_lamb(x[0], x[1]), x2d_lamb(x[0], x[1])])
        if np.linalg.norm(c_grad) < e1:
            break
        
        # Compute Hessian numerically
        H = np.array(hessian.subs({x1: x[0], x2: x[1]})).astype(float)
        
        # Check if Hessian is positive definite, if not, use gradient descent
        if np.linalg.det(H) <= 0 or np.any(np.linalg.eigvals(H) <= 0):
            d = -gamma * c_grad  # fallback to gradient descent
        else:
            H_inv = np.linalg.inv(H)
            d = -H_inv @ c_grad
        
        # Backtracking line search
        t = 1.0
        alpha = 0.1  # Armijo parameter
        beta = 0.5   # backtracking factor
        f_current = fun(x[0], x[1])
        grad_dot_d = c_grad @ d
        while t > 1e-10:
            x_next = x + t * d
            f_next = fun(x_next[0], x_next[1])
            if f_next <= f_current + alpha * t * grad_dot_d:
                break
            t *= beta
        
        x_new = x + t * d
        trajectory.append(x_new)
        values.append(fun(x_new[0], x_new[1]))
        
        # Check convergence
        if np.linalg.norm(x_new - x) < e2 and abs(values[-1] - values[-2]) < e2:
            break
        
        x = x_new
        k += 1

    return np.array(trajectory), np.array(values), k


x1, x2 = sp.symbols('x1, x2')
f = 100*(1 - x1**2 - x2**2)**2 + (x1-1)**2 + x2**2
np.random.seed(42)
p_list = (np.random.uniform(-3, 3, size=2), np.random.uniform(-3, 3, size=2), np.random.uniform(-3, 3, size=2))

x1_range = np.linspace(-10, 10, 100)
x2_range = np.linspace(-10, 10, 100)
x, y = np.meshgrid(x1_range, x2_range)
fu = 100*(1 - x**2 - y**2)**2 + (x-1)**2 + y**2
fig = go.Figure(data=[go.Contour(
    zmax = 5000, zmin = 0,
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
         coloring = 'heatmap',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig.update_layout(title=f'100(1-x1^2-x2^2)^2+(x1-1)^2+x2^2') 

for point in p_list:
    ans = newton(f, x0=point, e1 = 0.000001, e2 = 0.000001, gamma = 1, M = 50)
    print(ans[0][-1], ans[1][-1], ans[2])
    
    fig.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'точка {point.tolist()}'))
    fig.update_layout(
    legend=dict(
        orientation="h",   
        yanchor="bottom",
        y=1.02,            
        xanchor="right",
        x=1               
    )
)
fig.show()

[1.00000000e+00 5.09100122e-12] 2.6283974585882355e-23 22
[1.00000000e+00 4.34848607e-13] 1.1634787208929766e-24 11
[ 1.00000000e+00 -2.11716009e-12] 6.988042321367267e-24 26


In [ ]:
import numpy as np
import sympy as sp
import math
import plotly.graph_objects as go

def csgradM(f, x0, e1, e2, gamma, M):
    x = x0
    x1, x2 = sp.symbols('x1, x2')
    fun = sp.lambdify([x1, x2], f, "math")

    trajectory = [x0]
    values = [fun(x[0], x[1])]

    k = 0
    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    x1d = sp.lambdify([x1, x2], x1d, "math")
    x2d = sp.lambdify([x1, x2], x2d, "math")

    c = 0

    while k < M:
        c_grad = np.array([x1d(x[0], x[1]), x2d(x[0], x[1])])
        if np.linalg.norm(c_grad) < e1:
            break
        x_next = x - (gamma * c_grad)
        while fun(x_next[0], x_next[1]) - fun(x[0], x[1]) >= 0 and gamma > 1e-10:
            gamma = gamma / 2
            x_next = x - (gamma * c_grad)
        if np.linalg.norm(x_next - x) <= e2 and abs(fun(x_next[0], x_next[1]) - fun(x[0], x[1])) <= e2:
            c += 1
            if c == 2:
                break
        else:
            c = 0
            k += 1
            x = x_next
            trajectory.append(x)
            values.append(fun(x[0], x[1]))
    return np.array(trajectory), np.array(values), k

# ///////////// UNDER CONSTRUCTION ////////////// 
def qgradM(f, x0, e1, e2, gamma, M):
    x = x0.astype(float)
    x1, x2 = sp.symbols('x1, x2')
    alpha = sp.symbols('alpha')
    fun = sp.lambdify([x1, x2], f, "math")

    trajectory = [x.copy()]
    values = [fun(x[0], x[1])]

    k = 0
    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    x1d = sp.lambdify([x1, x2], x1d, "math")
    x2d = sp.lambdify([x1, x2], x2d, "math")

    c = 0

    while k < M:
        c_grad = np.array([x1d(x[0], x[1]), x2d(x[0], x[1])])
        if np.linalg.norm(c_grad) < e1:
            break

        d = -c_grad
        phi = f.subs({x1: x[0] + alpha * d[0], x2: x[1] + alpha * d[1]})
        alpha_candidates = sp.solve(sp.diff(phi, alpha), alpha)

        alpha_opt = None
        for cand in alpha_candidates:
            cand_val = complex(cand.evalf())
            if abs(cand_val.imag) < 1e-12 and cand_val.real > 0:
                alpha_opt = cand_val.real
                break
        if alpha_opt is None:
            alpha_opt = gamma

        x_next = x + alpha_opt * d
        if np.linalg.norm(x_next - x) <= e2 and abs(fun(x_next[0], x_next[1]) - fun(x[0], x[1])) <= e2:
            c += 1
            trajectory.append(x)
            values.append(fun(x[0], x[1]))
            if c == 2:
                break
        else:
            c = 0
            k += 1
            x = x_next
            trajectory.append(x)
            values.append(fun(x[0], x[1]))
    return np.array(trajectory), np.array(values), k

e_list = [10**(-2), 10**(-4), 10**(-6)]

x1, x2 = sp.symbols('x1, x2')
f = x1**2-x1*x2+x2**2+3*x1-2*x2+1
np.random.seed(42)
x0 = np.random.uniform(-10, 10, size=2)

x1_range = np.linspace(-10, 10, 100)
x2_range = np.linspace(-10, 10, 100)
x, y = np.meshgrid(x1_range, x2_range)
fu = x**2-x*y+y**2+3*x-2*y+1
fig = go.Figure(data=[go.Contour(
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
         coloring = 'heatmap',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig.update_layout(title=f'x1**2-x1*x2+x2**2+3*x1-2*x2+1') 

for e in e_list:
    print(e)
    ans = csgradM(f, x0=x0, e1 = e, e2 = e, gamma = 1, M = 50)
    print(ans[0][-1], ans[1][-1], ans[2])
    
    fig.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'для ε = {e}'))
    fig.update_layout(
    legend=dict(
        orientation="h",   
        yanchor="bottom",
        y=1.02,            
        xanchor="right",
        x=1               
    )
)
fig.show()

fig1 = go.Figure(data=[go.Contour(
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
         coloring = 'heatmap',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig1.update_layout(title=f'x1**2-x1*x2+x2**2+3*x1-2*x2+1') 

for e in e_list:
    print(e)
    ans = qgradM(f, x0=x0, e1 = e, e2 = e, gamma = 1, M = 50)
    print(ans[0][-1], ans[1][-1], ans[2])
    
    fig1.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'для ε = {e}'))
    fig1.update_layout(
    legend=dict(
        orientation="h",   
        yanchor="bottom",
        y=1.02,            
        xanchor="right",
        x=1               
    )
)
fig1.show()


0.01
[-1.32909459  0.33275918] -1.333312603023693 11
0.0001
[-1.3332671   0.33332436] -1.3333333282722228 17
1e-06
[-1.3333334   0.33333385] -1.3333333333330248 24


0.01
[-1.33374476  0.33637071] -1.3333226887848637 6
0.0001
[-1.33326784  0.33338861] -1.3333333296089185 9
1e-06
[-1.33333348  0.3333344 ] -1.33333333333203 12


In [1]:
import numpy as np
import sympy as sp
import plotly.graph_objects as go

def newton(f, x0, e1, e2, gamma, M):
    x = x0
    x1, x2 = sp.symbols('x1, x2')
    fun = sp.lambdify([x1, x2], f, "math")

    trajectory = [x0]
    values = [fun(x[0], x[1])]

    k = 0
    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    hessian = sp.hessian(f, [x1, x2])
    x1d = sp.lambdify([x1, x2], x1d, "math")
    x2d = sp.lambdify([x1, x2], x2d, "math")
    t = 1
    c = 0
    x = x.astype(float)
    flag = False
    while k < M:
        c_grad = np.array([x1d(x[0], x[1]), x2d(x[0], x[1])])
        if np.linalg.norm(c_grad) < e1:
            break
        H = np.array(hessian.subs({x1: x[0], x2: x[1]})).astype(float)
        
        if np.linalg.det(H) <= 0 or np.any(np.linalg.eigvals(H) <= 0):
            d = -gamma * c_grad 
        else:
            H_inv = np.linalg.inv(H)
            d = -H_inv @ c_grad
            Flag = True
        
        t = 1.0
        alpha = 0.1 
        beta = 0.5 
        f_current = fun(x[0], x[1])
        grad_dot_d = c_grad @ d

        if flag:
            flag = False
            continue
        else:
            while t > 1e-10:
                x_next = x + t * d
                f_next = fun(x_next[0], x_next[1])
                if f_next <= f_current + alpha * t * grad_dot_d:
                    break
                t *= beta

        x_next = x + t*d
        if np.linalg.norm(x_next - x) <= e2 and abs(fun(x_next[0], x_next[1]) - fun(x[0], x[1])) <= e2:
            c += 1
            trajectory.append(x)
            values.append(fun(x[0], x[1]))
            x = x_next
            if c == 2:
                break
        else:
            c = 0
            k += 1
            x = x_next
            trajectory.append(x)
            values.append(fun(x[0], x[1]))

    return np.array(trajectory), np.array(values), k


x1, x2 = sp.symbols('x1, x2')
f = 100*(1 - x1**2 - x2**2)**2 + (x1-1)**2 + x2**2
np.random.seed(42)
p_list = (np.random.uniform(-3, 3, size=2), np.random.uniform(-3, 3, size=2), np.random.uniform(-3, 3, size=2))

x1_range = np.linspace(-10, 10, 100)
x2_range = np.linspace(-10, 10, 100)
x, y = np.meshgrid(x1_range, x2_range)
fu = 100*(1 - x**2 - y**2)**2 + (x-1)**2 + y**2
fig = go.Figure(data=[go.Contour(
    zmax = 5000, zmin = 0,
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
         coloring = 'heatmap',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig.update_layout(title=f'100(1-x1^2-x2^2)^2+(x1-1)^2+x2^2') 

for point in p_list:
    print(f'point = {point}')
    ans = newton(f, x0=point, e1 = 0.000001, e2 = 0.000001, gamma = 1, M = 50)
    print(ans[0][-1], ans[1][-1], ans[2])
    
    fig.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'точка {point.tolist()}'))
    fig.update_layout(
    legend=dict(
        orientation="h",   
        yanchor="bottom",
        y=1.02,            
        xanchor="right",
        x=1               
    )
)
fig.show()

point = [-0.75275929  2.70428584]
[1.00000006e+00 2.25859727e-07] 1.3244360011287046e-12 22
point = [1.39196365 0.59195091]
[1.0000000e+00 3.1429785e-07] 1.0358068164706715e-13 11
point = [-2.06388816 -2.06403288]
[ 1.00000001e+00 -3.97659591e-07] 2.291748715090377e-13 26


In [ ]:
import numpy as np
import sympy as sp
import pandas as pd
import plotly.graph_objects as go
import random 


def auxiliary_point(grad, lower, upper, step=0.1, maxiter=500, eps=1e-5):
    x = np.array([(lower[0] + upper[0]) / 2, (lower[1] + upper[1]) / 2])
    for _ in range(maxiter):
        c_grad = np.array([grad[0](x[0], x[1]), grad[1](x[0], x[1])])
        x_new = x - step * c_grad   # шаг градиентного спуска
        x_new = np.clip(x_new, lower, upper)  # проекция на параллелепипед
        if np.linalg.norm(x_new - x) < eps:
            break
        x = x_new
    return x

def select_alpha_gd(f_sym, grad, x, p, alpha=0.5, step_size=0.1, maxiter=100, eps=1e-5):
    # Преобразуем f_sym в числовую функцию (аналогично auxiliary_point)
    f_num = sp.lambdify([x1, x2], f_sym, "math")
    
    for _ in range(maxiter):
        x_alpha = x + alpha * p
        # Вычисляем градиент в x_alpha через grad
        g = np.array([grad[0](x_alpha[0], x_alpha[1]),
                      grad[1](x_alpha[0], x_alpha[1])])
        grad_phi = np.dot(g, p)   # производная phi'(alpha)
        alpha_new = alpha - step_size * grad_phi
        alpha_new = np.clip(alpha_new, 0.0, 1.0)
        if np.linalg.norm(alpha_new - alpha) < eps:
            break
        alpha = alpha_new
    return alpha

def cond_grad(f, x0, lower, upper, eps = 1e-5, maxiter = 5000):
    x = x0.copy()
    
    if not lower[0]<= x[0] <= upper[0] and lower[1] <= x[1] <= upper[1]:
        return "Point is out of bounds"
    fun = sp.lambdify([x1, x2], f, "math")
    x_path = [x.copy()]
    fx = fun(x[0], x[1])
    f_path = [fx]

    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    x1d = sp.lambdify([x1, x2], x1d, "math")
    x2d = sp.lambdify([x1, x2], x2d, "math")

    for k in range(maxiter):

        x_bar = auxiliary_point([x1d, x2d], lower, upper)
        direction = x_bar - x
        if np.linalg.norm(direction) < eps:
            break

        alpha = select_alpha_gd(f, [x1d, x2d], x, direction, step_size=0.05, maxiter=50)

        x = x + alpha * direction
        x_path.append(x.copy())
        fx = fun(x[0], x[1])
        f_path.append(fx)

        
    return x_path, f_path, k+1


x1, x2 = sp.symbols('x1, x2')
f = (x1+x2)**4 + 4*x2**2
random.seed(42)
p_list = ([3, 3], [random.uniform(-2, 0), random.uniform(1, 3)], [random.uniform(-2, 0),  random.uniform(1, 3)])
e_list = (0.1, 1e-6)
lower = [-2, 1]
upper = [0, 3]
x1_range = np.linspace(-3, 1, 100)
x2_range = np.linspace(0, 4, 100)

x, y = np.meshgrid(x1_range, x2_range)
fu = (x+y)**4 + 4*y**2
fig = go.Figure(data=[go.Contour(
    zmax = 5000, zmin = 0,
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
         coloring = 'lines',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig.update_layout(title=f'(x1+x2)^4 + 4*x2^2') 
fig.add_shape(type="rect",
    x0=-2, y0=1, x1=0, y1=3,
    line=dict(color="RoyalBlue"),)

# method visualize

for point in p_list:
    for e in e_list:
        print(f'point = {point}')
        ans = cond_grad(f, point, lower, upper, eps=e)
        if isinstance(ans, str):
            print(ans)
            continue
        print(ans[0], ans[1], ans[2])
        fig.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'точка {point}'))
fig.show()

In [5]:
import numpy as np
import sympy as sp
import pandas as pd
import plotly.graph_objects as go
import random 



def auxiliary_point_exact(grad, x, lower, upper):
    g = np.array([grad[0](x[0], x[1]), grad[1](x[0], x[1])])
    return np.array([lower[i] if g[i] >= 0 else upper[i] for i in range(2)])

def cond_grad(f, x0, lower, upper, eps=1e-12, maxiter=5000):
    x = x0.copy()
    fun = sp.lambdify([x1, x2], f, "numpy")
    x_path = [x.copy()]
    f_path = [fun(x[0], x[1])]
    x1d = sp.lambdify([x1, x2], sp.diff(f, x1), "numpy")
    x2d = sp.lambdify([x1, x2], sp.diff(f, x2), "numpy")
    f_prev = float('inf')
    for k in range(maxiter):
        x_bar = auxiliary_point_exact([x1d, x2d], x, lower, upper)
        direction = x_bar - x
        norm_dir = np.linalg.norm(direction)
        fx = fun(x[0], x[1])
        if norm_dir < eps or abs(fx - f_prev) < 1e-12:
            break
        # Классический шаг Франк-Вулфа
        alpha = 2.0 / (k + 2)
        x = x + alpha * direction
        x_path.append(x.copy())
        f_prev = fx
        f_path.append(fun(x[0], x[1]))
    return x_path, f_path, k+1



x1, x2 = sp.symbols('x1, x2')
f = (x1+x2)**4 + 4*x2**2
random.seed(42)
p_list = ([-2, 3], [random.uniform(-2, 0), random.uniform(1, 3)], [random.uniform(-2, 0),  random.uniform(1, 3)])
e_list = (0.1, 1e-6)
lower = [-2, 1]
upper = [0, 3]
x1_range = np.linspace(-3, 1, 100)
x2_range = np.linspace(0, 4, 100)

x, y = np.meshgrid(x1_range, x2_range)
fu = (x+y)**4 + 4*y**2
fig = go.Figure(data=[go.Contour(
    zmax = 5000, zmin = 0,
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
         coloring = 'lines',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig.update_layout(title=f'(x1+x2)^4 + 4*x2^2') 
fig.add_shape(type="rect",
    x0=-2, y0=1, x1=0, y1=3,
    line=dict(color="RoyalBlue"),)

# method visualize

for point in p_list:
    for e in e_list:
        print(f'point = {point}')
        ans = cond_grad(f, point, lower, upper, eps=e)
        if isinstance(ans, str):
            print(ans)
            continue
        print(ans[0][-1], ans[1][-1], ans[2])
        fig.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'точка {point}'))
fig.show()

point = [-2, 3]
[-1.33333333  1.        ] 4.012345679012346 4
point = [-2, 3]
[-1.33333333  1.        ] 4.012345679012346 4
point = [-0.7211464030842325, 1.0500215104453339]
[-1.33333333  1.        ] 4.012345679012346 4
point = [-0.7211464030842325, 1.0500215104453339]
[-1.33333333  1.        ] 4.012345679012346 4
point = [-1.4499413632617615, 1.4464214762976455]
[-0.66666667  1.        ] 4.012345679012346 4
point = [-1.4499413632617615, 1.4464214762976455]
[-0.66666667  1.        ] 4.012345679012346 4


In [6]:
import numpy as np
import sympy as sp
import pandas as pd
import plotly.graph_objects as go
import random 


def auxiliary_point(grad, lower, upper, step_size=0.05, maxiter=10000, eps=1e-9):
    x = np.array([(lower[0] + upper[0]) / 2, (lower[1] + upper[1]) / 2])
    # x = np.array([lower[0], lower[1]])
    for _ in range(maxiter):
        c_grad = np.array([grad[0](x[0], x[1]), grad[1](x[0], x[1])])
        if np.linalg.norm(c_grad) < eps:
            break
        x_new = x - step_size * c_grad   # шаг градиентного спуска
        x_new = np.clip(x_new, lower, upper)  # проекция на параллелепипед
        if np.linalg.norm(x_new - x) < eps and np.linalg.norm():
            break
        x = x_new
    return x


def select_alpha_gd(grad, x, p, alpha=0.5, step_size=0.05, maxiter=10000, eps=1e-9):
    # perform gradient descent on scalar alpha for phi(alpha)=f(x+alpha*p)
    for _ in range(maxiter):
        x_alpha = x + alpha * p
        # Вычисляем градиент в x_alpha через grad
        g = np.array([grad[0](x_alpha[0], x_alpha[1]),
                      grad[1](x_alpha[0], x_alpha[1])])
        grad_a = np.dot(g, p)  # derivative d/dalpha f(x+alpha p)
        alpha_new = alpha - step_size * grad_a
        alpha_new = np.clip(alpha_new, 0.0, 1.0)
        if abs(alpha_new - alpha) < eps:
            break
        alpha = alpha_new
    return alpha


def cond_grad(f, x0, lower, upper, eps = 1e-5, maxiter = 20000, step_size=0.05):
    x = x0.copy()
    fun = sp.lambdify([x1, x2], f, "math")
    x_path = [x.copy()]
    fx = fun(x[0], x[1])
    f_path = [fx]

    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    x1d = sp.lambdify([x1, x2], x1d, "math")
    x2d = sp.lambdify([x1, x2], x2d, "math")

    for k in range(maxiter):
        x_bar = auxiliary_point(fun, [x1d, x2d], lower, upper, step_size = step_size, maxiter=10000)
        direction = x_bar - x
        if np.linalg.norm(direction) < eps:
            break

        alpha = select_alpha_gd([x1d, x2d], x, direction, step_size=step_size, maxiter=10000)

        x = x + alpha * direction
        x_path.append(x.copy())
        fx = fun(x[0], x[1])
        f_path.append(fx)

    return x_path, f_path, k


x1, x2 = sp.symbols('x1, x2')
f = (x1+x2)**4 + 4*x2**2
random.seed(42)
p_list = ([-2, 3], [random.uniform(-2, 0), random.uniform(1, 3)], [random.uniform(-2, 0),  random.uniform(1, 3)])
e_list = (0.1, 1e-6)
lower = [-2, 1]
upper = [0, 3]
x1_range = np.linspace(-3, 1, 100)
x2_range = np.linspace(0, 4, 100)

x, y = np.meshgrid(x1_range, x2_range)
fu = (x+y)**4 + 4*y**2
fig = go.Figure(data=[go.Contour(
    zmax = 5000, zmin = 0,
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
         coloring = 'lines',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig.update_layout(title=f'(x1+x2)^4 + 4*x2^2') 
fig.add_shape(type="rect",
    x0=-2, y0=1, x1=0, y1=3,
    line=dict(color="RoyalBlue"),)

# method visualize

for point in p_list:
    for e in e_list:
        print(f'point = {point}')
        ans = cond_grad(f, point, lower, upper, eps=e)
        print(ans[0][-1], ans[1][-1], ans[2])
        fig.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'точка {point}'))
fig.show()


point = [-2, 3]


TypeError: auxiliary_point() got multiple values for argument 'step_size'

In [7]:
import numpy as np
import sympy as sp
import pandas as pd
import plotly.graph_objects as go
import random 


def auxiliary_point_gd(grad, lower, upper, step_size=0.05, maxiter=10000, eps=1e-9):
    x = np.array([(lower[0] + upper[0]) / 2, (lower[1] + upper[1]) / 2])
    for _ in range(maxiter):
        c_grad = np.array([grad[0](x[0], x[1]), grad[1](x[0], x[1])])
        if np.linalg.norm(c_grad) < eps:
            break
        x_new = x - step_size * c_grad   # gradient descent step
        x_new = np.clip(x_new, lower, upper)  # project onto box
        if np.linalg.norm(x_new - x) < eps:
            break
        x = x_new
    return x


def select_alpha_gd(grad, x, p, alpha=0.5, step_size=0.05, maxiter=10000, eps=1e-9):
    # iterative search on alpha using directional derivative
    for _ in range(maxiter):
        x_alpha = x + alpha * p
        g = np.array([grad[0](x_alpha[0], x_alpha[1]),
                      grad[1](x_alpha[0], x_alpha[1])])
        grad_a = np.dot(g, p)
        alpha_new = alpha - step_size * grad_a
        alpha_new = np.clip(alpha_new, 0.0, 1.0)
        if abs(alpha_new - alpha) < eps:
            alpha = alpha_new
            break
        alpha = alpha_new
    return alpha


def estimate_lipschitz_hessian(f, x1, x2, lower, upper, samples=16):
    H = sp.hessian(f, (x1, x2))
    Hf = sp.lambdify([x1, x2], H, "numpy")
    sample_count = int(np.sqrt(samples))
    if sample_count < 2:
        sample_count = 2
    x1_vals = np.linspace(lower[0], upper[0], sample_count)
    x2_vals = np.linspace(lower[1], upper[1], sample_count)
    L_est = 0.0
    for a in x1_vals:
        for b in x2_vals:
            H_val = np.array(Hf(a, b), dtype=float)
            eigvals = np.linalg.eigvalsh(H_val)
            L_est = max(L_est, np.max(eigvals))
    return max(L_est, 1e-6)


def choose_step_size(eps, L_est):
    # smaller eps may require finer steps for stability,
    # but do not shrink below a safe lower bound.
    step = min(0.9 / L_est, max(0.005, np.sqrt(eps)))
    return step


def cond_grad(f, x0, lower, upper, eps=1e-5, maxiter=20000):
    x = x0.copy()
    fun = sp.lambdify([x1, x2], f, "math")
    x_path = [x.copy()]
    fx = fun(x[0], x[1])
    f_path = [fx]

    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    x1d = sp.lambdify([x1, x2], x1d, "math")
    x2d = sp.lambdify([x1, x2], x2d, "math")

    L_est = estimate_lipschitz_hessian(f, x1, x2, lower, upper, samples=16)
    step_size = choose_step_size(eps, L_est)

    for k in range(maxiter):
        x_bar = auxiliary_point_gd([x1d, x2d], lower, upper, step_size=step_size, maxiter=10000, eps=eps)
        direction = x_bar - x
        if np.linalg.norm(direction) < eps:
            break

        alpha = select_alpha_gd([x1d, x2d], x, direction, step_size=step_size, maxiter=10000, eps=eps)
        x = x + alpha * direction
        x_path.append(x.copy())
        fx = fun(x[0], x[1])
        f_path.append(fx)

    return x_path, f_path, k


x1, x2 = sp.symbols('x1, x2')
f = (x1+x2)**4 + 4*x2**2
random.seed(42)
p_list = ([-2, 3], [random.uniform(-2, 0), random.uniform(1, 3)], [random.uniform(-2, 0),  random.uniform(1, 3)])
e_list = (0.1, 1e-6)
lower = [-2, 1]
upper = [0, 3]
x1_range = np.linspace(-3, 1, 100)
x2_range = np.linspace(0, 4, 100)

x, y = np.meshgrid(x1_range, x2_range)
fu = (x+y)**4 + 4*y**2
fig = go.Figure(data=[go.Contour(
    zmax = 5000, zmin = 0,
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
         coloring = 'lines',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig.update_layout(title=f'(x1+x2)^4 + 4*x2^2') 
fig.add_shape(type="rect",
    x0=-2, y0=1, x1=0, y1=3,
    line=dict(color="RoyalBlue"),)

# method visualize

for point in p_list:
    for e in e_list:
        print(f'point = {point}')
        ans = cond_grad(f, point, lower, upper, eps=e)
        print(ans[0][-1], ans[1][-1], ans[2])
        fig.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'точка {point}'))
fig.show()


point = [-2, 3]
[-1.04717913  2.04717913] 17.76376963603496 4
point = [-2, 3]
[-1.04070017  1.        ] 4.000002744003946 1
point = [-0.7211464030842325, 1.0500215104453339]
[-0.97829963  1.92607272] 15.645920454835322 4
point = [-0.7211464030842325, 1.0500215104453339]
[-1.04070017  1.        ] 4.000002744003946 1
point = [-1.4499413632617615, 1.4464214762976455]
[-1.032502    1.96001165] 16.106654314516092 4
point = [-1.4499413632617615, 1.4464214762976455]
[-1.04070017  1.        ] 4.000002744003946 1


In [4]:
import numpy as np
import sympy as sp
import pandas as pd
import plotly.graph_objects as go
import random 


def auxiliary_point_gd(grad, lower, upper, step_size=0.05, maxiter=10000, eps=1e-9):
    x = np.array([(lower[0] + upper[0]) / 2, (lower[1] + upper[1]) / 2])
    # x = np.array([lower[0], lower[1]])
    for _ in range(maxiter):
        c_grad = np.array([grad[0](x[0], x[1]), grad[1](x[0], x[1])])
        if np.linalg.norm(c_grad) < eps:
            break
        x_new = x - step_size * c_grad   # шаг градиентного спуска
        x_new = np.clip(x_new, lower, upper)  # проекция на параллелепипед
        if np.linalg.norm(x_new - x) < eps:
            break
        x = x_new
    return x


def select_alpha_gd(grad, x, p, alpha=0.5, step_size=0.05, maxiter=1000, eps=1e-9):
    
    for _ in range(maxiter):
        x_alpha = x + alpha * p
        # Вычисляем градиент в x_alpha через grad
        g = np.array([grad[0](x_alpha[0], x_alpha[1]),
                      grad[1](x_alpha[0], x_alpha[1])])
        grad_a = np.dot(g, p) 
        alpha_new = alpha - step_size * grad_a
        alpha_new = np.clip(alpha_new, 0.0, 1.0)
        if abs(alpha_new - alpha) < eps:
            break
        alpha = alpha_new
    return alpha


def cond_grad(f, x0, lower, upper, eps=1e-5, maxiter=5000, step_size=0.05):
    x = x0.copy()
    fun = sp.lambdify([x1, x2], f, "math")
    x_path = [x.copy()]
    fx = fun(x[0], x[1])
    f_path = [fx]

    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    x1d = sp.lambdify([x1, x2], x1d, "math")
    x2d = sp.lambdify([x1, x2], x2d, "math")

    for k in range(maxiter):
        if fun(x[0], x[1]) > 0:
            x_bar = np.array([lower[0], lower[1]])
        else:
            x_bar = np.array([upper[0], upper[1]])
        direction = x_bar - x
        if np.linalg.norm(direction) < eps:
            break

        alpha = select_alpha_gd([x1d, x2d], x, direction, step_size=step_size, maxiter=10000)

        x = x + alpha * direction
        x_path.append(x.copy())
        fx = fun(x[0], x[1])
        f_path.append(fx)

    return x_path, f_path, k


x1, x2 = sp.symbols('x1, x2')
f = (x1+x2)**4 + 4*x2**2
random.seed(42)
p_list = ([-2, 3], [random.uniform(-2, 0),  random.uniform(1, 3)])
e_list = (0.1, 1e-6)
lower = [-2, 1]
upper = [0, 3]
x1_range = np.linspace(-3, 1, 100)
x2_range = np.linspace(0, 4, 100)

x, y = np.meshgrid(x1_range, x2_range)
fu = (x+y)**4 + 4*y**2
fig = go.Figure(data=[go.Contour(
    zmax = 5000, zmin = 0,
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
         coloring = 'lines',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig.update_layout(title=f'(x1+x2)^4 + 4*x2^2') 
fig.add_shape(type="rect",
    x0=-2, y0=1, x1=0, y1=3,
    line=dict(color="RoyalBlue"),)

# method visualize

for point in p_list:
    for e in e_list:
        print(f'point = {point}')
        ans = cond_grad(f, point, lower, upper, eps=e)
        print(ans[0][-1], ans[1][-1], ans[2])
        fig.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'точка {point}'))
fig.show()


point = [-2, 3]
[-2.  1.] 5.0 1
point = [-2, 3]
[-2.  1.] 5.0 1
point = [-0.7211464030842325, 1.0500215104453339]
[-1.44690516  1.02163394] 4.207652364900787 4999
point = [-0.7211464030842325, 1.0500215104453339]
[-1.44690516  1.02163394] 4.207652364900787 4999
